In [20]:
!pip install albumentations pillow faiss-cpu tqdm transformers torch

In [21]:
!pip install albumentations pillow faiss-cpu tqdm transformers torch

In [22]:
import time
import cv2
import os
import numpy as np
import faiss
import torch
import pickle
import gc
import random
from tqdm import tqdm
from PIL import Image
from transformers import ViTModel, AutoImageProcessor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import albumentations as A

import torch.nn as nn
import torch.nn.functional as F

class DinoClassifier(nn.Module):
    def __init__(self, num_classes):
        super(DinoClassifier, self).__init__()
        self.model = ViTModel.from_pretrained("facebook/dino-vits8")
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.model.config.hidden_size, num_classes)

    def forward(self, x):
        outputs = self.model(**x)
        cls_token = outputs.last_hidden_state[:, 0, :]
        out = self.dropout(cls_token)
        return self.fc(out)


def train_classifier(src_dir, num_classes, epochs=5, batch_size=16, lr=1e-4):
    from torchvision import transforms
    from torch.utils.data import DataLoader, Dataset

    class SimpleImageDataset(Dataset):
        def __init__(self, root_dir, class_to_id, transform=None):
            self.samples = []
            self.class_to_id = class_to_id
            self.transform = transform
            for label in os.listdir(root_dir):
                class_dir = os.path.join(root_dir, label)
                if not os.path.isdir(class_dir):
                    continue
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('jpg', 'jpeg', 'png')):
                        self.samples.append((os.path.join(class_dir, img_name), class_to_id[label]))

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, label = self.samples[idx]
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, label

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoImageProcessor.from_pretrained("facebook/dino-vits8")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    class_to_id = {label: idx for idx, label in enumerate(sorted(os.listdir(src_dir)))}
    dataset = SimpleImageDataset(src_dir, class_to_id, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = DinoClassifier(num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in dataloader:
            inputs = processor(images=list(images), return_tensors="pt").to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        acc = correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}, Accuracy: {acc:.4f}")

    return model, class_to_id


def evaluate_classifier(model, test_dir, class_to_id):
    from torchvision import transforms
    from torch.utils.data import DataLoader, Dataset

    class SimpleImageDataset(Dataset):
        def __init__(self, root_dir, class_to_id, transform=None):
            self.samples = []
            self.class_to_id = class_to_id
            self.transform = transform
            for label in os.listdir(root_dir):
                class_dir = os.path.join(root_dir, label)
                if not os.path.isdir(class_dir):
                    continue
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('jpg', 'jpeg', 'png')):
                        self.samples.append((os.path.join(class_dir, img_name), class_to_id[label]))

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, label = self.samples[idx]
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, label

    import torch
    from sklearn.metrics import classification_report

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = AutoImageProcessor.from_pretrained("facebook/dino-vits8")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    dataset = SimpleImageDataset(test_dir, class_to_id, transform=transform)
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

    model.eval()
    model.to(device)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            inputs = processor(images=list(images), return_tensors="pt").to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

    print(classification_report(all_labels, all_preds))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()


class DinoFaceRecognition:
        self.src_dir = src_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model_name = "facebook/dino-vits8"
        self.model = ViTModel.from_pretrained(self.model_name).to(self.device)
        self.model.eval()
        self.processor = AutoImageProcessor.from_pretrained(self.model_name)

        self.index = None
        self.labels = []
        self.class_to_id = {}
        self.id_to_class = {}

        self.augmentor = A.Compose([
            A.Resize(128, 128, always_apply=True),
            A.OneOf([
                A.CoarseDropout(min_holes=1, max_holes=1, min_height=10, max_height=20, min_width=10, max_width=20, fill_value=0, p=1.0),
                A.HorizontalFlip(p=1.0),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=1.0),
                A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=1.0)
            ], p=1.0)
        ])

    def augment_image(self, image):
        augmented = self.augmentor(image=image)
        return augmented['image']

    def load_data(self, faiss_index_path=None, metadata_path=None):
        if faiss_index_path and metadata_path:
            self.index = faiss.read_index(faiss_index_path)
            with open(metadata_path, 'rb') as f:
                metadata = pickle.load(f)
                self.labels = metadata['labels']
                self.class_to_id = metadata['label_to_id']
                self.id_to_class = {v: k for k, v in self.class_to_id.items()}

    def normalize_histogram(self, img):
        yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
        yuv[:, :, 0] = cv2.equalizeHist(yuv[:, :, 0])
        return cv2.cvtColor(yuv, cv2.COLOR_YUV2RGB)

    def extract_features(self, images, normalize=True):
        try:
            if not isinstance(images, list):
                images = [images]
            valid_images = []
            for img in images:
                if img is None:
                    continue
                if isinstance(img, np.ndarray):
                    if len(img.shape) == 3 and img.shape[2] == 3:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    else:
                        continue
                elif not isinstance(img, Image.Image):
                    continue
                img = self.normalize_histogram(img)
                valid_images.append(img)

            if not valid_images:
                return None

            inputs = self.processor(images=valid_images, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            if normalize:
                embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
            return embeddings
        except:
            return None

    def add_new_user_to_faiss(self, class_name, image_dir, faiss_index_path=None):
        if not os.path.exists(image_dir):
            print(f"[ERROR] Path {image_dir} does not exist")
            return

        images = []
        for img_name in os.listdir(image_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(image_dir, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img)

        if not images:
            print(f"[WARN] No valid images found in {image_dir}")
            return

        if class_name in self.class_to_id:
            class_id = self.class_to_id[class_name]
        else:
            class_id = len(self.class_to_id)
            self.class_to_id[class_name] = class_id
            self.id_to_class[class_id] = class_name

        features = self.extract_features(images)
        if features is None:
            print("[ERROR] Failed to extract features")
            return

        if self.index is None:
            dim = features.shape[1]
            self.index = faiss.IndexFlatIP(dim)

        self.index.add(features.astype('float16'))
        self.labels.extend([class_id] * len(features))

        mean_vec = np.mean(features, axis=0, keepdims=True)
        self.index.add(mean_vec.astype('float16'))
        self.labels.append(class_id)

        num_aug = max(1, int(len(images) * 0.2))
        selected_imgs = random.sample(images, num_aug)
        aug_images = [self.augment_image(img) for img in selected_imgs]
        aug_features = self.extract_features(aug_images)
        if aug_features is not None:
            self.index.add(aug_features.astype('float16'))
            self.labels.extend([class_id] * len(aug_features))

        print(f"[ADD USER] {class_name}: Added {len(features)} original, {len(aug_features)} aug, 1 mean vector")
        print(f"[FAISS] Total vectors: {self.index.ntotal}, Total labels: {len(self.class_to_id)}")

        self.save_index(faiss_index_path)


    def save_index(self, save_path):
        if self.index is not None and save_path:
            faiss.write_index(self.index, save_path + '.faiss')
            with open(f"{save_path}_metadata.pkl", "wb") as f:
                pickle.dump({
                    "labels": self.labels,
                    "label_to_id": self.class_to_id
                }, f)

    def train_in_batches(self, faiss_index_path=None, batch_size=5):
        if not self.src_dir:
            raise ValueError("Source directory (src_dir) not specified")

        self.labels = []
        all_class_names = sorted(os.listdir(self.src_dir))
        total_labels = len(all_class_names)

        for i in tqdm(range(0, total_labels, batch_size)):
            batch_class_names = all_class_names[i:i + batch_size]
            print(f"[Batch {i // batch_size + 1}] Training on classes: {batch_class_names} ({i + len(batch_class_names)} / {total_labels})")

            for class_name in batch_class_names:
                class_dir = os.path.join(self.src_dir, class_name)
                images = []
                if os.path.isdir(class_dir):
                    for img_name in os.listdir(class_dir):
                        img_path = os.path.join(class_dir, img_name)
                        if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                            img = cv2.imread(img_path)
                            if img is None:
                                continue
                            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                            images.append(img)

                if not images:
                    continue

                if class_name not in self.class_to_id:
                    class_id = len(self.class_to_id)
                    self.class_to_id[class_name] = class_id
                    self.id_to_class[class_id] = class_name
                else:
                    class_id = self.class_to_id[class_name]

                features = self.extract_features(images)
                if features is None:
                    continue
                if self.index is None:
                    dim = features.shape[1]
                    self.index = faiss.IndexFlatIP(dim)

                # Add all original features
                self.index.add(features.astype('float16'))
                self.labels.extend([class_id] * len(features))

                # Add mean feature
                mean_vec = np.mean(features, axis=0, keepdims=True)
                self.index.add(mean_vec.astype('float16'))
                self.labels.append(class_id)

                # Randomly select 20% images for augmentation
                num_aug = max(1, int(len(images) * 0.2))
                selected_imgs = random.sample(images, num_aug)
                aug_images = [self.augment_image(img) for img in selected_imgs]
                aug_features = self.extract_features(aug_images)
                if aug_features is not None:
                    self.index.add(aug_features.astype('float16'))
                    self.labels.extend([class_id] * len(aug_features))

                print(f"[INFO] Class {class_name}: Original images: {len(images)}, Augmented: {len(aug_features)}, Total vectors added: {len(images) + 1 + len(aug_features)}")
                print(f"[FAISS] Total vectors so far: {self.index.ntotal}")
                print(f"[FAISS] Total labels so far: {len(self.class_to_id)}")

            self.save_index(faiss_index_path)
            gc.collect()
            torch.cuda.empty_cache()


In [23]:
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

dino = DinoFaceRecognition(src_dir='train')
dino.train_in_batches(faiss_index_path='faiss_index_no_augment', batch_size=1)

  0%|          | 0/962 [00:00<?, ?it/s]

[Batch 1] Training on classes: ['n000001'] (1 / 962)


  0%|          | 1/962 [00:00<12:39,  1.27it/s]

[INFO] Class n000001: Original images: 100, Augmented: 20, Total vectors added: 121
[FAISS] Total vectors so far: 121
[FAISS] Total labels so far: 1
[Batch 2] Training on classes: ['n000009'] (2 / 962)


  0%|          | 2/962 [00:01<13:17,  1.20it/s]

...

100%|█████████▉| 960/962 [12:43<00:01,  1.17it/s]

[Batch 961] Training on classes: ['n009291'] (961 / 962)
[INFO] Class n009291: Original images: 100, Augmented: 20, Total vectors added: 121
[FAISS] Total vectors so far: 119069
[FAISS] Total labels so far: 961


100%|█████████▉| 961/962 [12:44<00:00,  1.18it/s]

[Batch 962] Training on classes: ['n009294'] (962 / 962)
[INFO] Class n009294: Original images: 100, Augmented: 20, Total vectors added: 121
[FAISS] Total vectors so far: 119190
[FAISS] Total labels so far: 962


100%|██████████| 962/962 [12:45<00:00,  1.26it/s]


In [24]:
dino.load_data('faiss_index_no_augment.faiss', 'faiss_index_no_augment_metadata.pkl')

In [25]:

dino.print_faiss_size('faiss_index_no_augment.faiss')
#|%%--%%| <AGMxKnSMeM|mJYsVXolyl>
if dino.index is not None:
    print(f"[INFO] FAISS Index contains {dino.index.ntotal} vectors.")
else:
    print("[WARN] FAISS index not initialized.")


[INFO] FAISS Index size: 174.59 MB (183075885 bytes)
[INFO] FAISS Index contains 119190 vectors.


In [ ]:
results = dino.evaluate_on_testset(test_dir='test')
print(results)

pred_label: n004237 | true_label: n001299 | score: 0.8204189538955688
pred_label: n007594 | true_label: n001299 | score: 0.7481534481048584
pred_label: n004338 | true_label: n001299 | score: 0.6771526336669922
pred_label: n007210 | true_label: n001299 | score: 0.739985466003418
pred_label: n002836 | true_label: n001299 | score: 0.8529106378555298
pred_label: n009246 | true_label: n001299 | score: 0.8514645099639893
pred_label: n006245 | true_label: n001299 | score: 0.7345873117446899
pred_label: n002080 | true_label: n001299 | score: 0.8535455465316772
pred_label: n004276 | true_label: n001299 | score: 0.7858808636665344
pred_label: n006874 | true_label: n001299 | score: 0.8293551206588745
pred_label: n007865 | true_label: n001299 | score: 0.7773402333259583
pred_label: n001241 | true_label: n001299 | score: 0.7177127599716187
pred_label: n004826 | true_label: n001299 | score: 0.7411367893218994
pred_label: n006643 | true_label: n001299 | score: 0.7401552200317383
pred_label: n008659 |